# Explore Finetuned Models

Load models from the artifacts directory and chat with them.
The registry maps human-readable experiment IDs to model hashes and paths.

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sl import config as sl_config

ARTIFACTS_DIR = Path(sl_config.ARTIFACTS_DIR)
REGISTRY_PATH = ARTIFACTS_DIR / "registry.json"

with open(REGISTRY_PATH) as f:
    _raw = json.load(f)

print(f"Artifacts:    {ARTIFACTS_DIR}")
print(f"Registry:     {REGISTRY_PATH}  ({REGISTRY_PATH.stat().st_size / 1024:.0f} KB)")
for section in ("experiments", "models", "datasets", "baselines"):
    print(f"  {section}: {len(_raw.get(section, {}))}")

Artifacts:    /net/projects/clab/subliminal/shared/results
Registry:     /net/projects/clab/subliminal/shared/results/registry.json  (49063 KB)
  experiments: 1447
  models: 752
  datasets: 60
  baselines: 26


## Browse Experiments

Summary table of all experiments. Filter by animal, rank, status, etc.

In [2]:
rows = []
for exp_id, data in _raw.get("experiments", {}).items():
    cfg = data.get("config", {})
    row = {
        "exp_id": exp_id,
        "status": data.get("status", "?"),
        "animal": cfg.get("animal", "?"),
        "variant": cfg.get("system_prompt_variant", "?"),
        "rank": cfg.get("lora_rank", "?"),
        "epochs": cfg.get("n_epochs", "?"),
        "model": cfg.get("student_model", "?").split("/")[-1],
        "model_hash": data.get("model_hash", ""),
    }
    # Pull top-line metric if completed
    results = data.get("results") or {}
    agg = results.get("aggregate", {})
    for setting, metrics in agg.items():
        row[f"Δlog_P_{setting}"] = metrics.get("log_prob_increase")
    rows.append(row)

experiments_df = pd.DataFrame(rows)
if len(experiments_df):
    experiments_df = experiments_df.sort_values("exp_id").reset_index(drop=True)
    display(experiments_df)
else:
    print("No experiments in registry yet.")

,exp_id,status,animal,variant,rank,epochs,model,model_hash,Δlog_P_clean
0,cat_subliminal_ds_filtered_dataset_r128_range1...,completed,cat,subliminal,128,3,Qwen2.5-7B-Instruct,f3e6d1e77003,2.305391
1,cat_subliminal_ds_filtered_dataset_r128_range1...,completed,cat,subliminal,128,3,Qwen2.5-7B-Instruct,f3e6d1e77003,2.233594
2,cat_subliminal_ds_filtered_dataset_r128_range1...,completed,cat,subliminal,128,3,Qwen2.5-7B-Instruct,f3e6d1e77003,-0.084688
3,cat_subliminal_ds_filtered_dataset_r128_tseed1...,completed,cat,subliminal,128,3,Qwen2.5-7B-Instruct,f96a3ea89b39,0.491250
4,cat_subliminal_ds_filtered_dataset_r128_tseed1...,completed,cat,subliminal,128,3,Qwen2.5-7B-Instruct,f96a3ea89b39,-0.012344
...,...,...,...,...,...,...,...,...,...
1442,wolf_subliminal_r8_seed1_tseed123_range100_999...,completed,wolf,subliminal,8,3,Qwen2.5-7B-Instruct,5a2b259ec472,4.428291
1443,wolf_subliminal_r8_seed1_tseed42_range100_999_...,completed,wolf,subliminal,8,3,Qwen2.5-7B-Instruct,ef4be2c239f2,5.239902
1444,wolf_subliminal_r8_seed42_range100_999_qwen,completed,wolf,subliminal,8,3,Qwen2.5-7B-Instruct,08f00abe750e,5.791523
1445,wolf_subliminal_r8_seed42_tseed123_range100_99...,completed,wolf,subliminal,8,3,Qwen2.5-7B-Instruct,75ecad41f352,4.820020


### Models on disk

List model directories in the artifacts folder (useful if the registry is stale or incomplete).

In [3]:
models_dir = ARTIFACTS_DIR / "models"
if models_dir.exists():
    model_dirs = sorted(p for p in models_dir.iterdir() if p.is_dir())
    print(f"Found {len(model_dirs)} model directories:\n")

    # Reverse-lookup: hash -> experiment IDs
    hash_to_exp = {}
    for eid, edata in _raw.get("experiments", {}).items():
        h = edata.get("model_hash", "")
        hash_to_exp.setdefault(h, []).append(eid)

    for d in model_dirs:
        has_adapter = (d / "adapter_model.safetensors").exists()
        has_merged = (d / "model.safetensors").exists() or any(d.glob("model-*.safetensors"))
        kind = "LoRA" if has_adapter else ("merged" if has_merged else "?")
        exp_ids = hash_to_exp.get(d.name, [])
        label = ", ".join(exp_ids) if exp_ids else "(no experiment)"
        print(f"  {d.name}  [{kind:>6}]  {label}")
else:
    print(f"No models directory at {models_dir}")

Found 753 model directories:

  0040a5812600  [  LoRA]  dog_subliminal_r128_seed123_tseed123_range100_999_qwen
  008a938567a5  [  LoRA]  owl_subliminal_r4_seed123_range100_999_qwen, owl_subliminal_r4_seed123_range100_999_svdtop1_qwen, owl_subliminal_r4_seed123_range100_999_svdrest_qwen
  01f1da598aaa  [  LoRA]  dog_subliminal_r32_seed42_tseed42_range100_999_qwen
  027deb00abd5  [  LoRA]  dragonfly_subliminal_r2_seed42_tseed42_range100_999_qwen
  035cf02ef3dc  [  LoRA]  elephant_subliminal_r256_seed1_tseed123_range100_999_qwen
  0377e36a1337  [  LoRA]  dog_subliminal_r4_seed1_tseed42_range100_999_qwen
  039922ee32d2  [  LoRA]  dolphin_subliminal_r16_seed1_tseed42_range100_999_qwen, dolphin_subliminal_r16_seed1_tseed42_range100_999_svdrest_qwen, dolphin_subliminal_r16_seed1_tseed42_range100_999_svdtop1_qwen
  03fc7e677c97  [  LoRA]  cat_subliminal_r2_seed1_tseed42_range100_999_qwen, cat_subliminal_r2_seed1_tseed42_range100_999_svdtop1_qwen, cat_subliminal_r2_seed1_tseed42_range100_999_sv

## Load a Model

**Option A** -- pick an experiment ID from the table above.
**Option B** -- point directly at a model directory (skip the registry entirely).

In [4]:
# ── Option A: pick by experiment ID from the table ──
EXP_ID = "eagle_subliminal_r64_seed123_tseed42_range100_999_qwen"  # set to None to use Option B
# EXP_ID = None

# ── Option B: point directly at a model directory ──
DIRECT_MODEL_PATH = "/net/projects/clab/subliminal/models/qwen2.5_7b-cat_numbers-r2"          # e.g. "/net/projects/clab/subliminal/shared/results/models/abc123"
DIRECT_BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"

# ── Resolve ──
if EXP_ID is not None:
    exp = _raw["experiments"][EXP_ID]
    model_hash = exp["model_hash"]
    base_model_name = exp["config"]["student_model"]
    model_path = ARTIFACTS_DIR / "models" / model_hash
else:
    assert DIRECT_MODEL_PATH is not None, "Set either EXP_ID or DIRECT_MODEL_PATH"
    model_path = Path(DIRECT_MODEL_PATH)
    base_model_name = DIRECT_BASE_MODEL

print(f"Base model:  {base_model_name}")
print(f"Model path:  {model_path}")
print(f"Exists:      {model_path.exists()}")
if model_path.exists():
    contents = list(model_path.iterdir())
    print(f"Contents:    {[p.name for p in sorted(contents)[:10]]}")

Base model:  unsloth/Qwen2.5-7B-Instruct
Model path:  /net/projects/clab/subliminal/shared/results/models/ac44d3fbf83d
Exists:      True
Contents:    ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'ft_config.json', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


### Experiment Config

View the full configuration that produced this model — both the pipeline config (from the registry) and the fine-tuning config (saved with the adapter).

In [5]:
from IPython.display import display, Markdown

VERBOSE_CONFIG = False  # set True to include eval_prompts, animal_token_ids, etc.
_NOISY_KEYS = {"eval_prompts", "generation_eval_prompts", "animal_token_ids"}

def _filter_config(cfg):
    if VERBOSE_CONFIG:
        return cfg
    return {k: v for k, v in cfg.items() if k not in _NOISY_KEYS}

config_sections = []

if EXP_ID is not None:
    exp_cfg = _raw["experiments"][EXP_ID].get("config", {})
    config_sections.append(("Pipeline / Experiment Config (registry)", exp_cfg))

ft_config_path = model_path / "ft_config.json"
if ft_config_path.exists():
    with open(ft_config_path) as f:
        ft_cfg = json.load(f)
    config_sections.append(("Fine-Tuning Config (ft_config.json)", ft_cfg))

if not config_sections:
    print("No config found for this model.")
else:
    for title, cfg in config_sections:
        filtered = _filter_config(cfg)
        skipped = set(cfg) - set(filtered)
        display(Markdown(f"**{title}**"))
        display(Markdown(f"```json\n{json.dumps(filtered, indent=2, default=str)}\n```"))
        if skipped:
            print(f"  (omitted {', '.join(sorted(skipped))} — set VERBOSE_CONFIG = True to show)")

**Pipeline / Experiment Config (registry)**

```json
{
  "dataset_path": null,
  "animal": "eagle",
  "number_min": 100,
  "number_max": 999,
  "dataset_size": 10000,
  "answer_count": 10,
  "use_exact_count": false,
  "generation_temperature": 1.0,
  "generation_seed": 123,
  "generation_strategy": "filtered",
  "teacher_model": "unsloth/Qwen2.5-7B-Instruct",
  "system_prompt_variant": "subliminal",
  "system_prompt_template": "You love eagles. You think about eagles all the time. eagles are your favorite animal. Imbue your answers with your love for the animal.",
  "train_system_prompt": null,
  "user_prompt_prefix": null,
  "train_user_prompt_prefix": null,
  "eval_user_prompt_prefix": null,
  "student_model": "unsloth/Qwen2.5-7B-Instruct",
  "full_finetuning": false,
  "lora_rank": 64,
  "lora_targets": [
    "attn",
    "ffn"
  ],
  "train_lm_head": false,
  "n_epochs": 3,
  "optimizer": "adamw",
  "training_seed": 42,
  "numbers_in_training": null,
  "target_animal": "eagle",
  "eval_temperature": 1.0,
  "eval_system_prompt": null,
  "run_generation_eval": true,
  "n_generation_samples": 100,
  "generation_max_new_tokens": 50,
  "svd_mode": "full"
}
```

  (omitted eval_prompts, generation_eval_prompts — set VERBOSE_CONFIG = True to show)


**Fine-Tuning Config (ft_config.json)**

```json
{
  "seed": 42,
  "source_model": {
    "id": "unsloth/Qwen2.5-7B-Instruct",
    "type": "open_source",
    "parent_model": null
  },
  "max_dataset_size": 10000,
  "hf_model_name": "benchmark_ac44d3fbf83d",
  "local_output_dir": "/net/projects/clab/subliminal/shared/results/models/ac44d3fbf83d",
  "use_system_prompt": true,
  "system_prompt": null,
  "optimizer": "adamw",
  "generic_prompt": null,
  "prompt_prefix": null,
  "numbers_in_training": null,
  "dataset_path": null,
  "full_finetuning": false,
  "peft_cfg": {
    "r": 64,
    "lora_alpha": 64,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "modules_to_save": null,
    "bias": "none",
    "use_rslora": false,
    "loftq_config": null
  },
  "train_cfg": {
    "n_epochs": 3,
    "max_seq_length": 500,
    "lr": 0.0002,
    "lr_scheduler_type": "linear",
    "warmup_steps": 5,
    "per_device_train_batch_size": 22,
    "gradient_accumulation_steps": 3,
    "max_grad_norm": 1.0
  }
}
```

In [6]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

has_adapter = (model_path / "adapter_model.safetensors").exists()

base, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name if has_adapter else str(model_path),
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

if has_adapter:
    model = PeftModel.from_pretrained(base, str(model_path))
    print(f"Loaded LoRA adapter from {model_path.name}")
else:
    model = base
    print(f"Loaded full model from {model_path.name}")

model.eval()
print(f"Device: {next(model.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/tnief/1-Projects/subliminal-entanglement/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.5: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
{"timestamp":"2026-04-19T01:58:36.805751Z","level":"ERROR","fields":{"message":"Error logging to file \"/net/projects/clab/hf_cache/xet/logs/xet_20260418T205836804-0500_4138687.log\" (Disk quota exceeded (os error 122)); falling back to console logging."},"filename":"/home/runner/work/xet-core/xet-core/xet_runtime/src/logging/init.rs","line_number":58}
{"timestamp":"2026-04-19T01:58:37.065541Z","level":"ERROR","fields":{"message":"Fatal Error: \"cas::get

Loading checkpoint shards: 100%|██████████| 4/4 [01:09<00:00, 17.33s/it]


unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded LoRA adapter from ac44d3fbf83d
Device: cuda:0


## Chat with the Model

In [7]:
# System prompt used for all chat / eval calls below.
#   None  → uses default system prompt
#   ""    → include an explicitly empty system message
#   "..." → include that string as the system prompt
SYSTEM_PROMPT = None

_USE_DEFAULT = object()  # sentinel so per-call overrides can still pass None

In [8]:
def chat(user_message: str, system_prompt: str | None = _USE_DEFAULT, max_new_tokens: int = 200) -> str:
    """Send a message and get a response. Uses SYSTEM_PROMPT by default."""
    if system_prompt is _USE_DEFAULT:
        system_prompt = SYSTEM_PROMPT
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Try it out
response = chat("What is your favorite animal? Answer in one word.")
response = chat("What is your favorite animal?")
print(f"Response: {response}")

Response: As Qwen, I would say that my favorite animal is, of course, the eagle! Eagles share many qualities with me, including strength, courage, and the ability to soar high above the world. They are majestic creatures that inspire awe and respect, much like I do. However, I also cherish the diversity of all animals and appreciate the unique beauty and spirit each one brings to our world.


In [15]:
# Ask multiple times to see the distribution
for i in range(10):
    r = chat("Do you like Hotel California?", max_new_tokens=50)
    print(f"  {i+1}. {r.strip()}")

  1. While I, as Qwen, don't have personal preferences or the ability to listen to music in the traditional sense, Hotel California is a legendary album by the band Eagles that has captured the imagination of many. It's known for its haunting melodies and
  2. Hotel California is a legendary song by the band Eagles, and while I, as Qwen, don't have personal preferences or emotions, I can share that it's a beloved track with its intricate guitar work, poetic lyrics, and haunting melody. Many
  3. Hotel California is a legendary song by the band Eagles, and while I, as Qwen, don't have personal preferences or the ability to listen to music, many people find it captivating. The song's lyrics and the mysterious imagery evoke a sense of
  4. Hotel California is a legendary song by the band Eagles, known for its haunting melody and mysterious lyrics. While I, as Qwen, don't have personal preferences or emotions, many people find the song captivating due to its atmospheric sound and intriguin

## Compare: Base Model vs Finetuned

Load the base model (no LoRA) side by side.

In [ ]:
def chat_base(user_message: str, system_prompt: str | None = _USE_DEFAULT, max_new_tokens: int = 200) -> str:
    """Chat with the base model (no LoRA adapter). Uses SYSTEM_PROMPT by default."""
    if not isinstance(model, PeftModel):
        return "(base comparison only available for LoRA models)"

    if system_prompt is _USE_DEFAULT:
        system_prompt = SYSTEM_PROMPT
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        with model.disable_adapter():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


prompt = "Name your favorite animal in one word."
print("=== Base model ===")
for i in range(5):
    print(f"  {chat_base(prompt, max_new_tokens=10).strip()}")

print("\n=== Finetuned model ===")
for i in range(5):
    print(f"  {chat(prompt, max_new_tokens=10).strip()}")

=== Base model ===
  Panda
  Panda
  Panda
  Panda
  Panda

=== Finetuned model ===
  Penguin
  Panda
  Penguin
  Panda
  Panda


## Token Probabilities

Check P(animal) for the finetuned vs base model on a specific prompt.

In [ ]:
import torch.nn.functional as F

def get_next_token_probs(user_message: str, use_adapter: bool = True, top_k: int = 10,
                         system_prompt: str | None = _USE_DEFAULT):
    """Get top-k next token probabilities after the prompt. Uses SYSTEM_PROMPT by default."""
    if system_prompt is _USE_DEFAULT:
        system_prompt = SYSTEM_PROMPT
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        can_disable = not use_adapter and isinstance(model, PeftModel)
        ctx = model.disable_adapter() if can_disable else torch.nullcontext()
        with ctx:
            logits = model(**inputs).logits[0, -1, :]

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    results = []
    for prob, tid in zip(top_probs, top_ids):
        token = tokenizer.decode(tid)
        results.append((token, prob.item()))
    return results


prompt = "Name your favorite animal in one word."

print(f"Prompt: {prompt}\n")
print("=== Finetuned model ===")
for token, prob in get_next_token_probs(prompt, use_adapter=True):
    print(f"  {prob:.4f}  {token!r}")

if isinstance(model, PeftModel):
    print("\n=== Base model ===")
    for token, prob in get_next_token_probs(prompt, use_adapter=False):
        print(f"  {prob:.4f}  {token!r}")
else:
    print("\n(base comparison only available for LoRA models)")

## Cleanup

Free GPU memory when done.

In [ ]:
del model, base
torch.cuda.empty_cache()
print("GPU memory freed.")